# ADRMiner — LLM-based MADR Adherence Checking

## Purpose

This notebook implements the **ADR checking** stage of the ADRMiner workflow. It uses an
LLM-as-a-judge strategy to evaluate whether Architecture Decision Records (ADRs) follow
key elements of the MADR template.

For each ADR, the checker evaluates:

- overall MADR adherence;
- section presence;
- alternative section titles;
- content quality; and
- purpose consistency.

The analyzed sections are **Context**, **Decision**, **Consequences**,
**Decision Drivers**, and **Considered Options**.

The notebook supports the results reported for **RQ3** in the companion paper,
*“A Text Mining and Classification Approach for Analyzing Architecture Decision
Records.”*

## Supported LLM providers

The notebook can use either:

- **OpenAI**, through the standard OpenAI API; or
- **Ollama**, through Ollama's OpenAI-compatible endpoint.

Both providers are instantiated through LangChain's `ChatOpenAI` interface so that the
rest of the ADR checking pipeline remains unchanged.

## Main inputs

- `../data/LLM4ADR-adrs__adrs_english.pickle`: curated ADR corpus.
- Provider-specific model configuration in the repository `.env` file.

## Main outputs

- `../results/all_projects-checks_results.json`: aggregated ADR-checking results.
- Optional partial and batch JSON files under `../results/`.
- A structured JSON assessment for each analyzed ADR.

## Important execution notes

- Each ADR check triggers several LLM calls because MADR sections are evaluated
  independently.
- Full-corpus execution may take a long time.
- OpenAI execution may incur API costs.
- Ollama execution requires a running local Ollama server and a model that supports
  reliable structured output.
- The artifact includes precomputed results. Keep `FORCE_RECOMPUTE = False` to reuse
  them when available.
- Run this notebook from the `notebooks/` directory so that relative paths resolve
  correctly.


## 1. Environment and dependencies

The following cell imports the Python dependencies and ADRMiner modules used by the
checking workflow.

In [1]:
from tqdm import tqdm

import re
import json
import logging
import os
import pickle
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import display, Markdown
from langchain_openai import ChatOpenAI

import utils
from adr_checking import ADRChecker
from adr import adr

## 2. Configuration

Configure the input dataset, output path, LLM provider, model, cache policy, and batch
range here.

### OpenAI configuration

To use OpenAI, define the following variables in the repository `.env` file:

```text
LLM_PROVIDER=openai
OPENAI_API_KEY=your-api-key
OPENAI_MODEL_NAME=gpt-4.1-mini
```

### Ollama configuration

To use a local Ollama model through its OpenAI-compatible API, first start Ollama and
pull the desired model:

```bash
ollama serve
ollama pull qwen3:8b
```

Then configure:

```text
LLM_PROVIDER=ollama
OLLAMA_MODEL_NAME=qwen3:8b
OLLAMA_BASE_URL=http://localhost:11434/v1
```

Local Ollama does not require authentication. However, `ChatOpenAI` expects an API-key
value, so the notebook supplies the placeholder value `ollama`; Ollama ignores it.

> The checker expects structured JSON responses. Select an Ollama model with adequate
> instruction-following and structured-output capabilities. Different local models may
> produce results that differ from those reported in the paper.

### Result caching

- `FORCE_RECOMPUTE = False` reuses an existing result file.
- `FORCE_RECOMPUTE = True` reruns the LLM checks and overwrites cached results.

### Batch range

The original study divided the corpus into batches because checking every ADR requires
many model calls. `BATCH_START` and `BATCH_END` define the slice of valid projects
processed in this notebook. Set `BATCH_START = 0` and `BATCH_END = None` to process all
valid projects in one run.


In [2]:
# Input dataset and main output file.
DATASET_PATH = '../data/LLM4ADR-adrs__adrs_english.pickle'
ALL_PROJECTS_RESULTS = './results/all_projects-checks_results.json'

# Reuse cached outputs by default. Set to True to rerun the LLM checks and
# overwrite an existing aggregate result file.
FORCE_RECOMPUTE = False

# Long executions can be divided into project slices.
# Use BATCH_START = 0 and BATCH_END = None to process all valid projects.
BATCH_START = 216
BATCH_END = 312

# Save an intermediate JSON file after this many projects.
SAVE_PARTIAL_EVERY = 10

# ADRChecker can process ADRs within a project concurrently.
PARALLEL_CHECKING = True

# Example project used in the single-ADR demonstration.
EXAMPLE_ORGANIZATION = 'SAP'
EXAMPLE_PROJECT = 'cloud-sdk-js'
EXAMPLE_ADR_INDEX = 8

# Set to True only when intentionally merging previously generated batch files.
SAVE_MERGED_RESULTS = False

In [3]:
# ------------------------------------------------------------------
# LLM provider configuration
# ------------------------------------------------------------------
# Supported values: "openai" or "ollama".
# Values in .env override the default below.

load_dotenv()
LLM_PROVIDER = os.getenv('LLM_PROVIDER', 'openai').strip().lower()

if LLM_PROVIDER == 'openai':
    LLM_MODEL_NAME = os.getenv('OPENAI_MODEL_NAME')
    LLM_API_KEY = os.getenv('OPENAI_API_KEY')
    LLM_BASE_URL = None

    missing_variables = [
        name
        for name, value in {
            'OPENAI_API_KEY': LLM_API_KEY,
            'OPENAI_MODEL_NAME': LLM_MODEL_NAME,
        }.items()
        if not value
    ]
    if missing_variables:
        raise EnvironmentError(
            'Missing required OpenAI variable(s): '
            + ', '.join(missing_variables)
            + '. Configure them in the repository .env file.'
        )

elif LLM_PROVIDER == 'ollama':
    LLM_MODEL_NAME = os.getenv('OLLAMA_MODEL_NAME')
    LLM_BASE_URL = os.getenv(
        'OLLAMA_BASE_URL',
        'http://localhost:11434/v1',
    ).rstrip('/')

    # Local Ollama ignores this value, but ChatOpenAI requires a non-empty key.
    LLM_API_KEY = os.getenv('OLLAMA_API_KEY', 'ollama')

    if not LLM_MODEL_NAME:
        raise EnvironmentError(
            'OLLAMA_MODEL_NAME is not configured. Add it to the repository '
            '.env file, for example: OLLAMA_MODEL_NAME=qwen3:8b'
        )

else:
    raise ValueError(
        f"Unsupported LLM_PROVIDER={LLM_PROVIDER!r}. "
        "Use 'openai' or 'ollama'."
    )

In [4]:
def slugify(value: str) -> str:
    """Convert a provider/model name into a filename-safe suffix."""
    return re.sub(r'[^a-zA-Z0-9._-]+', '-', value).strip('-').lower()

# Keep archived paper filenames unchanged for the default OpenAI configuration.
# Ollama outputs receive a provider/model suffix by default.
default_suffix = (
    ''
    if LLM_PROVIDER == 'openai'
    else f"-ollama-{slugify(LLM_MODEL_NAME)}"
)
RESULTS_SUFFIX = os.getenv('RESULTS_SUFFIX', default_suffix)

def with_results_suffix(path: Path) -> Path:
    """Append the experiment suffix before a file extension."""
    if not RESULTS_SUFFIX:
        return path
    return path.with_name(f'{path.stem}{RESULTS_SUFFIX}{path.suffix}')

ALL_PROJECTS_RESULTS = with_results_suffix(Path(ALL_PROJECTS_RESULTS))

In [5]:

print('LLM provider:', LLM_PROVIDER)
print('LLM model:', LLM_MODEL_NAME)
print('LLM base URL:', LLM_BASE_URL or 'OpenAI default')
print('Force recomputation:', FORCE_RECOMPUTE)
print('Configured project slice:', BATCH_START, BATCH_END)

LLM provider: ollama
LLM model: llama3:8b
LLM base URL: http://localhost:11434/v1
Force recomputation: False
Configured project slice: 216 312


## 3. Initialize the language model

The following cell creates a `ChatOpenAI` instance for the selected provider:

- for OpenAI, it uses the standard API and configured API key;
- for Ollama, it redirects `ChatOpenAI` to the local OpenAI-compatible endpoint.

Temperature is set to zero to reduce output variability. The Responses API is disabled
explicitly so that both providers use the chat-completions interface.


In [6]:
# Create one ChatOpenAI-compatible client for either provider.
#
# Ollama exposes an OpenAI-compatible /v1/chat/completions endpoint, so the
# same LangChain interface can be reused by redirecting base_url.

llm_kwargs = {
    'model': LLM_MODEL_NAME,
    'api_key': LLM_API_KEY,
    'temperature': 0.0,
    'use_responses_api': False,
}

if LLM_BASE_URL:
    llm_kwargs['base_url'] = LLM_BASE_URL

llm = ChatOpenAI(**llm_kwargs)

print(
    f"Initialized {LLM_PROVIDER} model {LLM_MODEL_NAME!r}"
    + (f" at {LLM_BASE_URL}" if LLM_BASE_URL else "")
)


Initialized ollama model 'llama3:8b' at http://localhost:11434/v1


### 3.1 Optional provider smoke test

Run the following cell before launching the ADR checks. It verifies that the selected
provider is reachable and that the configured model can return a basic response.

For Ollama, a failure commonly means that:

- `ollama serve` is not running;
- the configured model has not been pulled; or
- `OLLAMA_BASE_URL` is incorrect.

This test does not guarantee that the model will follow the complete ADR checking schema,
but it catches basic connection and model-name problems early.


In [7]:
# Set to False if you prefer to skip the connectivity test.
RUN_LLM_SMOKE_TEST = True

if RUN_LLM_SMOKE_TEST:
    try:
        smoke_response = llm.invoke(
            'Reply with exactly the word READY.'
        )
        print('Provider response:', smoke_response.content)
    except Exception as exc:
        raise RuntimeError(
            f'Unable to invoke the configured {LLM_PROVIDER} model '
            f'{LLM_MODEL_NAME!r}. Check the provider settings and service.'
        ) from exc


Provider response: READY


## 4. ADR corpus

## 4. Load and filter the ADR corpus

The curated dataset is organized by organization, project, and ADR. We reproduce the
study's project-level inclusion criteria by retaining projects with at least five ADRs
and an average ADR length of at least 500 characters.

In [8]:
# Resolve the curated dataset relative to the notebooks/ directory.
with open(DATASET_PATH, 'rb') as file:
    dict_adrs = pickle.load(file)

print(len(dict_adrs), 'top-level organizations/projects loaded')

# Reproduce the project-level inclusion criteria used in the study.
projects_dict = utils.process_projects(
    dict_adrs,
    min_adrs_per_project=5,
    min_adr_length=500,
)
valid_projects = projects_dict['valid_projects']

docs = utils.get_documents(
    valid_projects,
    dict_adrs,
    field='raw',
)
print(len(valid_projects), 'valid organization/project pairs')
print(len(docs), 'ADRs considered within valid projects')


547 top-level organizations/projects loaded


organizations:   0%|          | 0/547 [00:00<?, ?it/s]

312 valid organization/project pairs
4316 ADRs considered within valid projects


## 5. Select an example project

This cell loads the ADRs belonging to one organization/project pair. Change
`EXAMPLE_ORGANIZATION` and `EXAMPLE_PROJECT` to inspect another project.

In [9]:
# Load all ADRs for the configured example organization/project pair.
adrs_to_check = utils.get_documents_by_key(
    (EXAMPLE_ORGANIZATION, EXAMPLE_PROJECT),
    dict_adrs,
    field='raw',
)

print(
    len(adrs_to_check),
    f"ADRs loaded for {EXAMPLE_ORGANIZATION}/{EXAMPLE_PROJECT}",
)


30 ADRs loaded for SAP/cloud-sdk-js


## 6. Initialize the ADR checker

`ADRChecker` encapsulates the prompts, structured output schema, section-level checks,
batch processing, and result persistence.

In [10]:
# Create the reusable LLM-based MADR checker.
checker = ADRChecker(llm)


## 7. Check a single ADR

This example runs the complete checking workflow on one ADR and displays both the
structured JSON assessment and the original Markdown document.

> **Cost note:** A single call to `checker.check(...)` may trigger several LLM requests
> because each MADR section is analyzed independently.

In [11]:
# Select one ADR from the example project.
example_adrs = list(adrs_to_check.values())
if not example_adrs:
    raise ValueError(
        f"No ADRs found for {EXAMPLE_ORGANIZATION}/{EXAMPLE_PROJECT}."
    )

if EXAMPLE_ADR_INDEX >= len(example_adrs):
    raise IndexError(
        f"EXAMPLE_ADR_INDEX={EXAMPLE_ADR_INDEX} is out of range for "
        f"{len(example_adrs)} available ADRs."
    )

example_adr = example_adrs[EXAMPLE_ADR_INDEX]

# Run the complete MADR adherence and section-consistency assessment.
# This may trigger several LLM calls internally.
result = checker.check(
    example_adr,
    metadata={
        'organization': EXAMPLE_ORGANIZATION,
        'project': EXAMPLE_PROJECT,
    },
)

print(json.dumps(result, indent=4))
display(Markdown(example_adr))


{"asctime": "2026-06-20 09:30:43,745", "levelname": "INFO", "name": "adr_checking", "message": "Classifying ADR with 298 tokens."}
{"asctime": "2026-06-20 09:31:59,690", "levelname": "INFO", "name": "adr_checking", "message": "Classifying ADR with 298 tokens."}
{
    "section_assessments": [
        {
            "section_name": "Context",
            "presence": "No",
            "content_quality": "Yes",
            "purpose_consistency": "Partial",
            "justification": "The section is missing, but the content fulfilling this role appears under 'Motivation'. The motivation provides a clear background and problem statement. However, it also includes some solution-related information, which makes it partially overlap with the 'Considered Options' or 'Decision' sections.",
            "alternate_title": [
                "Motivation"
            ]
        },
        {
            "section_name": "Decision",
            "presence": "Yes",
            "content_quality": "Yes",
   

## Motivation

Typescript allows for variadic functions e.g.:

```
function doSomething(...strings:string[]){
}
```

You can call the method with no, one, two, ... arguments.
We used this feature quite regularly for example for the `and` and `or` filter functions.
In the TypeScript universe there is also nothing wrong wit it, because you will get a type error if you call the function with an array.

```
function doSomething(['a','b','c']) //type error
```

However, in the JavaScript use case you do not get a type error if you call it with an array only a strange error at runtime.
From the method signature it also looks like an array is a valid input.
Hence, we decided to be lenient to the users and allow also for:

```
doSomething([])
doSomething([a])
doSomething([a,b])
```

in addition to the already possible:

```
doSomething()
doSomething('a')
doSomething('a','b')
```

## Solution

We use method overloading:

```
function functionWithVariableArguments(...varargs: string[]);
function functionWithVariableArguments(array: string[]);
function functionWithVariableArguments(
firstOrArray: undefined | string | string[],
...rest: string[]
): string[] {

}
```

and created a little helper method `variableArgumentToArray()` to merge the two argument `first` and `rest` to an array.


## 8. Apply the checker to a batch of projects

This stage checks the ADRs in the configured project slice and stores the results as
JSON. Partial files are written periodically to reduce the risk of losing progress
during a long execution.

If the aggregated output already exists and `FORCE_RECOMPUTE` is `False`, the notebook
loads the cached results instead of making new API calls.

In [12]:
# Reduce checker logging noise during batch execution.
logging.getLogger('adr_checking').setLevel(level=logging.CRITICAL)

relative_path = os.path.join('..', ALL_PROJECTS_RESULTS)
file_exists = os.path.isfile(relative_path)
should_recompute = FORCE_RECOMPUTE or not file_exists

if should_recompute:
    if FORCE_RECOMPUTE and file_exists:
        print(
            'FORCE_RECOMPUTE is enabled; cached aggregate results '
            'will be overwritten.'
        )

    selected_projects = valid_projects[BATCH_START:BATCH_END]
    print(
        f'Checking {len(selected_projects)} project(s) from the configured '
        f'slice [{BATCH_START}:{BATCH_END}].'
    )

    all_results = []

    for batch_index, (org, project) in enumerate(
        tqdm(selected_projects, desc='Processing organizations/projects'),
        start=1,
    ):
        adrs_to_check = utils.get_documents_by_key(
            (org, project),
            dict_adrs,
            field='raw',
        )

        project_results = checker.check_batch(
            adrs_to_check,
            parallel=PARALLEL_CHECKING,
            project=project,
            organization=org,
            as_dict=True,
        )
        all_results.extend(project_results)

        # Persist intermediate results periodically during long executions.
        if SAVE_PARTIAL_EVERY and batch_index % SAVE_PARTIAL_EVERY == 0:
            partial_path = relative_path.replace(
                '.json',
                (
                    f'_partial_{BATCH_START}_{BATCH_END}_'
                    f'{batch_index}.json'
                ),
            )
            print('Saving partial results to', partial_path)
            ADRChecker.save_results(all_results, partial_path)

    # Save the results produced by this configured project slice.
    batch_path = relative_path.replace(
        '.json',
        f'_batch_{BATCH_START}_{BATCH_END}.json',
    )
    print('Saving batch results to', batch_path)
    ADRChecker.save_results(all_results, batch_path)

    # Preserve the original notebook behavior by also writing the aggregate path.
    # When processing several slices, use the merge section below to construct the
    # complete aggregate file after all batches have finished.
    print('Saving current results to', relative_path)
    ADRChecker.save_results(all_results, relative_path)

else:
    with open(relative_path, 'r', encoding='utf-8') as file:
        all_results = json.load(file)

    print(
        'Loaded cached results from', relative_path,
        '\nSet FORCE_RECOMPUTE=True to regenerate them.',
    )

print('End.')


Checking 96 project(s) from the configured slice [216:312].


Processing organizations/projects:   9%|▉         | 9/96 [1:40:06<16:07:46, 667.43s/it]


LengthFinishReasonError: Could not parse response content as the length limit was reached - CompletionUsage(completion_tokens=40960, prompt_tokens=3881, total_tokens=44841, completion_tokens_details=None, prompt_tokens_details=None)

In [13]:
# Confirm how many ADR-level assessments are available in memory.
print(len(all_results), 'ADRs processed.')


166 ADRs processed.


## 9. Batch-result aggregation

When the complete study is executed as several project slices, each run may create a
file whose name contains `batch`. This section discovers those files and combines their
contents.

Set `SAVE_MERGED_RESULTS = True` only after confirming that the listed batch files are
the intended inputs. The merged file overwrites `ALL_PROJECTS_RESULTS`.

In [14]:
# Locate JSON files produced by separate batch executions.
results_dir = os.path.join('..', 'results')
batch_files = sorted(
    filename
    for filename in os.listdir(results_dir)
    if 'batch' in filename and filename.endswith('.json')
)

print('Batch files found:')
for filename in batch_files:
    print(' -', filename)

merged_results = []
for filename in batch_files:
    batch_path = os.path.join(results_dir, filename)
    with open(batch_path, 'r', encoding='utf-8') as file:
        merged_results.extend(json.load(file))

print(len(merged_results), 'ADR assessments collected from batch files')

merged_path = os.path.join('..', ALL_PROJECTS_RESULTS)
if SAVE_MERGED_RESULTS:
    print('Saving merged results to', merged_path)
    ADRChecker.save_results(merged_results, merged_path)
else:
    print(
        'Merged results were not written. '
        'Set SAVE_MERGED_RESULTS=True after validating the batch-file list.'
    )


Batch files found:
 - all_projects-checks_results_batch_0_156.json
 - all_projects-checks_results_batch_156_216.json
 - all_projects-checks_results_batch_216_312.json
4209 ADR assessments collected from batch files
Merged results were not written. Set SAVE_MERGED_RESULTS=True after validating the batch-file list.


## 10. Expected artifacts and next steps

After completing this notebook, the primary artifact is:

- `../results/all_projects-checks_results.json`

Each record contains the LLM-generated MADR adherence and section-level assessments for
one ADR. These outputs can be inspected directly or combined with topic and taxonomy
classification results in the downstream analysis notebooks.